# 02 - Data Preprocessing
### Customer Churn Prediction in the Banking Sector

This notebook cleans and transforms the raw data into a model-ready dataset, following
Table 2 of the reference paper:

| Feature | Transformation |
|---|---|
| CLIENTNUM | Dropped (not used in prediction) |
| Attrition_Flag | Label encoding (target) |
| Customer_Age | Standardization |
| Gender | Label encoding |
| Dependent_count | Unchanged |
| Education_Level | One-hot encoding |
| Marital_Status | One-hot encoding |
| Income_Category | One-hot encoding |
| Card_Category | One-hot encoding |
| Months_on_book | Standardization |
| Total_Relationship_Count | Unchanged |
| Months_Inactive_12_mon | Unchanged |
| Contacts_Count_12_mon | Unchanged |
| Credit_Limit | Normalization |
| Total_Revolving_Bal | Normalization |
| Avg_Open_To_Buy | Normalization |
| Total_Amt_Chng_Q4_Q1 | Unchanged |
| Total_Trans_Amt | Normalization |
| Total_Trans_Ct | Normalization |
| Total_Ct_Chng_Q4_Q1 | Unchanged |
| Avg_Utilization_Ratio | Unchanged |

Steps:
1. Load raw data, drop unused columns
2. Handle `Unknown` values
3. Clean categorical values (Divorced → Single, College → Graduate, as specified in the paper)
4. Label encode binary categoricals (`Gender`, `Attrition_Flag`)
5. One-hot encode multi-class categoricals
6. Standardize `Customer_Age`, `Months_on_book`
7. Normalize the wide-range quantitative features
8. Save the final processed dataset for clustering & modeling


In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder

pd.set_option('display.max_columns', None)


## 1. Load Raw Data & Drop Unused Columns

In [2]:
DATA_PATH = "../data/raw/BankChurners.csv"

df = pd.read_csv(DATA_PATH)
print("Original shape:", df.shape)

# Drop the 2 Naive-Bayes helper columns Kaggle auto-generates
nb_cols = [c for c in df.columns if 'Naive_Bayes' in c]
df = df.drop(columns=nb_cols)

# CLIENTNUM is an identifier, not a predictive feature
df = df.drop(columns=['CLIENTNUM'])

print("Shape after dropping unused columns:", df.shape)
df.head()


Original shape: (10127, 23)
Shape after dropping unused columns: (10127, 20)


,Attrition_Flag,Customer_Age,Gender,Dependent_count,Education_Level,Marital_Status,Income_Category,Card_Category,Months_on_book,Total_Relationship_Count,Months_Inactive_12_mon,Contacts_Count_12_mon,Credit_Limit,Total_Revolving_Bal,Avg_Open_To_Buy,Total_Amt_Chng_Q4_Q1,Total_Trans_Amt,Total_Trans_Ct,Total_Ct_Chng_Q4_Q1,Avg_Utilization_Ratio
0,Existing Customer,45,M,3,High School,Married,$60K - $80K,Blue,39,5,1,3,12691.0,777,11914.0,1.335,1144,42,1.625,0.061
1,Existing Customer,49,F,5,Graduate,Single,Less than $40K,Blue,44,6,1,2,8256.0,864,7392.0,1.541,1291,33,3.714,0.105
2,Existing Customer,51,M,3,Graduate,Married,$80K - $120K,Blue,36,4,1,0,3418.0,0,3418.0,2.594,1887,20,2.333,0.000
3,Existing Customer,40,F,4,High School,Unknown,Less than $40K,Blue,34,3,4,1,3313.0,2517,796.0,1.405,1171,20,2.333,0.760
4,Existing Customer,40,M,3,Uneducated,Married,$60K - $80K,Blue,21,5,1,0,4716.0,0,4716.0,2.175,816,28,2.500,0.000


## 2. Handle 'Unknown' Values

In [3]:
unknown_counts = (df == 'Unknown').sum()
unknown_counts = unknown_counts[unknown_counts > 0]
print("Columns with 'Unknown' values:")
print(unknown_counts)

# Drop rows containing any 'Unknown' value (as per the paper's "Eliminate unknown values" step)
before = len(df)
df = df[~(df == 'Unknown').any(axis=1)].reset_index(drop=True)
after = len(df)

print(f"\nRows before: {before}, after dropping Unknowns: {after} (removed {before - after})")


Columns with 'Unknown' values:
Education_Level    1519
Marital_Status      749
Income_Category    1112
dtype: int64

Rows before: 10127, after dropping Unknowns: 7081 (removed 3046)


## 3. Clean Categorical Values

As specified in the paper: replace `Divorced` → `Single`, and `College` → `Graduate`.


In [4]:
print("Marital_Status before:", df['Marital_Status'].unique())
print("Education_Level before:", df['Education_Level'].unique())

df['Marital_Status'] = df['Marital_Status'].replace('Divorced', 'Single')
df['Education_Level'] = df['Education_Level'].replace('College', 'Graduate')

print("\nMarital_Status after:", df['Marital_Status'].unique())
print("Education_Level after:", df['Education_Level'].unique())


Marital_Status before: ['Married' 'Single' 'Divorced']
Education_Level before: ['High School' 'Graduate' 'Uneducated' 'College' 'Post-Graduate'
 'Doctorate']

Marital_Status after: ['Married' 'Single']
Education_Level after: ['High School' 'Graduate' 'Uneducated' 'Post-Graduate' 'Doctorate']


## 4. Label Encoding (Binary Categorical Features)

In [5]:
le_gender = LabelEncoder()
le_target = LabelEncoder()

df['Gender'] = le_gender.fit_transform(df['Gender'])              # F=0, M=1 (alphabetical)
df['Attrition_Flag'] = le_target.fit_transform(df['Attrition_Flag'])  # Attrited=0, Existing=1

print("Gender mapping:", dict(zip(le_gender.classes_, le_gender.transform(le_gender.classes_))))
print("Attrition_Flag mapping:", dict(zip(le_target.classes_, le_target.transform(le_target.classes_))))

# NOTE: We want churn (Attrited Customer) to be the positive class (1) for clarity in modeling.
# le_target currently maps Attrited Customer -> 0, Existing Customer -> 1 (alphabetical order).
# Flip it so churn = 1.
df['Attrition_Flag'] = 1 - df['Attrition_Flag']
print("\nAfter flip -> churn (Attrited Customer) = 1, stayed (Existing Customer) = 0")
print(df['Attrition_Flag'].value_counts())


Gender mapping: {'F': np.int64(0), 'M': np.int64(1)}
Attrition_Flag mapping: {'Attrited Customer': np.int64(0), 'Existing Customer': np.int64(1)}

After flip -> churn (Attrited Customer) = 1, stayed (Existing Customer) = 0
Attrition_Flag
0    5968
1    1113
Name: count, dtype: int64


## 5. One-Hot Encoding (Multi-Class Categorical Features)

In [6]:
onehot_cols = ['Education_Level', 'Marital_Status', 'Income_Category', 'Card_Category']

df = pd.get_dummies(df, columns=onehot_cols, drop_first=False)

print("Shape after one-hot encoding:", df.shape)
df.head()


Shape after one-hot encoding: (7081, 32)


,Attrition_Flag,Customer_Age,Gender,Dependent_count,Months_on_book,Total_Relationship_Count,Months_Inactive_12_mon,Contacts_Count_12_mon,Credit_Limit,Total_Revolving_Bal,Avg_Open_To_Buy,Total_Amt_Chng_Q4_Q1,Total_Trans_Amt,Total_Trans_Ct,Total_Ct_Chng_Q4_Q1,Avg_Utilization_Ratio,Education_Level_Doctorate,Education_Level_Graduate,Education_Level_High School,Education_Level_Post-Graduate,Education_Level_Uneducated,Marital_Status_Married,Marital_Status_Single,Income_Category_$120K +,Income_Category_$40K - $60K,Income_Category_$60K - $80K,Income_Category_$80K - $120K,Income_Category_Less than $40K,Card_Category_Blue,Card_Category_Gold,Card_Category_Platinum,Card_Category_Silver
0,0,45,1,3,39,5,1,3,12691.0,777,11914.0,1.335,1144,42,1.625,0.061,False,False,True,False,False,True,False,False,False,True,False,False,True,False,False,False
1,0,49,0,5,44,6,1,2,8256.0,864,7392.0,1.541,1291,33,3.714,0.105,False,True,False,False,False,False,True,False,False,False,False,True,True,False,False,False
2,0,51,1,3,36,4,1,0,3418.0,0,3418.0,2.594,1887,20,2.333,0.000,False,True,False,False,False,True,False,False,False,False,True,False,True,False,False,False
3,0,40,1,3,21,5,1,0,4716.0,0,4716.0,2.175,816,28,2.500,0.000,False,False,False,False,True,True,False,False,False,True,False,False,True,False,False,False
4,0,44,1,2,36,3,1,2,4010.0,1247,2763.0,1.376,1088,24,0.846,0.311,False,True,False,False,False,True,False,False,True,False,False,False,True,False,False,False


## 6. Standardization

`Customer_Age` and `Months_on_book` look approximately normally distributed (seen in EDA)
→ apply **StandardScaler** (zero mean, unit variance).


In [7]:
standardize_cols = ['Customer_Age', 'Months_on_book']

scaler_std = StandardScaler()
df[standardize_cols] = scaler_std.fit_transform(df[standardize_cols])

df[standardize_cols].describe()


,Customer_Age,Months_on_book
count,7.081000e+03,7.081000e+03
mean,4.816559e-17,-2.809659e-17
std,1.000071e+00,1.000071e+00
min,-2.530601e+00,-2.871936e+00
25%,-6.650813e-01,-6.225108e-01
50%,-4.324162e-02,2.329584e-03
75%,7.029661e-01,5.022019e-01
max,3.314693e+00,2.501691e+00


## 7. Normalization

Wide-range / skewed quantitative features → apply **MinMaxScaler** (scale to [0, 1]).


In [8]:
normalize_cols = [
    'Credit_Limit', 'Total_Revolving_Bal', 'Avg_Open_To_Buy',
    'Total_Trans_Amt', 'Total_Trans_Ct'
]

scaler_norm = MinMaxScaler()
df[normalize_cols] = scaler_norm.fit_transform(df[normalize_cols])

df[normalize_cols].describe()


,Credit_Limit,Total_Revolving_Bal,Avg_Open_To_Buy,Total_Trans_Amt,Total_Trans_Ct
count,7081.000000,7081.000000,7081.000000,7081.000000,7081.000000
mean,0.213270,0.463846,0.212160,0.222150,0.439543
std,0.275898,0.322732,0.264573,0.198368,0.192011
min,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.032037,0.183949,0.036073,0.090306,0.274194
50%,0.086121,0.509337,0.094080,0.189934,0.459677
75%,0.280875,0.707588,0.274911,0.241922,0.564516
max,1.000000,1.000000,1.000000,1.000000,1.000000


## 8. Unchanged Features (Sanity Check)

These features have narrow variance and are left as-is:
`Dependent_count`, `Total_Relationship_Count`, `Months_Inactive_12_mon`,
`Contacts_Count_12_mon`, `Total_Amt_Chng_Q4_Q1`, `Total_Ct_Chng_Q4_Q1`,
`Avg_Utilization_Ratio`.


In [9]:
unchanged_cols = [
    'Dependent_count', 'Total_Relationship_Count', 'Months_Inactive_12_mon',
    'Contacts_Count_12_mon', 'Total_Amt_Chng_Q4_Q1', 'Total_Ct_Chng_Q4_Q1',
    'Avg_Utilization_Ratio'
]
df[unchanged_cols].describe()


,Dependent_count,Total_Relationship_Count,Months_Inactive_12_mon,Contacts_Count_12_mon,Total_Amt_Chng_Q4_Q1,Total_Ct_Chng_Q4_Q1,Avg_Utilization_Ratio
count,7081.000000,7081.000000,7081.000000,7081.000000,7081.000000,7081.000000,7081.000000
mean,2.337805,3.819376,2.342607,2.454456,0.760584,0.711508,0.282313
std,1.291649,1.544444,0.995104,1.104917,0.223139,0.238693,0.278731
min,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,1.000000,3.000000,2.000000,2.000000,0.629000,0.583000,0.026000
50%,2.000000,4.000000,2.000000,2.000000,0.735000,0.700000,0.186000
75%,3.000000,5.000000,3.000000,3.000000,0.858000,0.818000,0.515000
max,5.000000,6.000000,6.000000,6.000000,3.397000,3.714000,0.999000


## 9. Final Check & Save Processed Data

In [10]:
print("Final shape:", df.shape)
print("\nAny remaining nulls?", df.isnull().sum().sum())
print("\nColumn dtypes:\n", df.dtypes.value_counts())
df.head()


Final shape: (7081, 32)

Any remaining nulls? 0

Column dtypes:
 bool       16
float64    10
int64       6
Name: count, dtype: int64


,Attrition_Flag,Customer_Age,Gender,Dependent_count,Months_on_book,Total_Relationship_Count,Months_Inactive_12_mon,Contacts_Count_12_mon,Credit_Limit,Total_Revolving_Bal,Avg_Open_To_Buy,Total_Amt_Chng_Q4_Q1,Total_Trans_Amt,Total_Trans_Ct,Total_Ct_Chng_Q4_Q1,Avg_Utilization_Ratio,Education_Level_Doctorate,Education_Level_Graduate,Education_Level_High School,Education_Level_Post-Graduate,Education_Level_Uneducated,Marital_Status_Married,Marital_Status_Single,Income_Category_$120K +,Income_Category_$40K - $60K,Income_Category_$60K - $80K,Income_Category_$80K - $120K,Income_Category_Less than $40K,Card_Category_Blue,Card_Category_Gold,Card_Category_Platinum,Card_Category_Silver
0,0,-0.167610,1,3,0.377234,5,1,3,0.340190,0.308701,0.345116,1.335,0.036260,0.258065,1.625,0.061,False,False,True,False,False,True,False,False,False,True,False,False,True,False,False,False
1,0,0.329862,0,5,1.002074,6,1,2,0.206112,0.343266,0.214093,1.541,0.044667,0.185484,3.714,0.105,False,True,False,False,False,False,True,False,False,False,False,True,True,False,False,False
2,0,0.578598,1,3,0.002330,4,1,0,0.059850,0.000000,0.098948,2.594,0.078753,0.080645,2.333,0.000,False,True,False,False,False,True,False,False,False,False,True,False,True,False,False,False
3,0,-0.789449,1,3,-1.872192,5,1,0,0.099091,0.000000,0.136557,2.175,0.017501,0.145161,2.500,0.000,False,False,False,False,True,True,False,False,False,True,False,False,True,False,False,False
4,0,-0.291978,1,2,0.002330,3,1,2,0.077747,0.495431,0.079970,1.376,0.033057,0.112903,0.846,0.311,False,True,False,False,False,True,False,False,True,False,False,False,True,False,False,False


In [11]:
import os
os.makedirs("../data/processed", exist_ok=True)

OUTPUT_PATH = "../data/processed/churn_processed.csv"
df.to_csv(OUTPUT_PATH, index=False)
print(f"Processed dataset saved to: {OUTPUT_PATH}")
print("Shape:", df.shape)


Processed dataset saved to: ../data/processed/churn_processed.csv
Shape: (7081, 32)


## Summary

- Dropped `CLIENTNUM` and Kaggle's auto-generated Naive-Bayes helper columns.
- Removed rows containing `Unknown` values.
- Cleaned `Marital_Status` (Divorced → Single) and `Education_Level` (College → Graduate).
- Label-encoded `Gender` and `Attrition_Flag` (target: **1 = churn, 0 = stayed**).
- One-hot encoded `Education_Level`, `Marital_Status`, `Income_Category`, `Card_Category`.
- Standardized `Customer_Age`, `Months_on_book`.
- Normalized `Credit_Limit`, `Total_Revolving_Bal`, `Avg_Open_To_Buy`, `Total_Trans_Amt`,
  `Total_Trans_Ct`.
- Saved the final dataset to `data/processed/churn_processed.csv`.

➡️ **Next:** `03_clustering.ipynb` — apply K-means clustering (with the elbow method) on
this processed data to perform customer segmentation.
